In [9]:
import pandas as pd
import requests
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()

# Project pathsx
DATA_RAW = Path("../data/raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)

print("Setup complete. Data will be saved to:", DATA_RAW.resolve())

Setup complete. Data will be saved to: /Users/ramyashreeb/Projects/explainable-ai-grid/data/raw


In [10]:
# Quick test pull — current intensity
r = requests.get("https://api.carbonintensity.org.uk/intensity")
print(r.status_code)
print(r.json())

200
{'data': [{'from': '2026-07-10T07:30Z', 'to': '2026-07-10T08:00Z', 'intensity': {'forecast': 186, 'actual': 176, 'index': 'high'}}]}


In [11]:
frames = []

for year in range(2021, 2024):
    for month in range(1, 13):
        url = f"https://api.carbonintensity.org.uk/intensity/{year}-{month:02d}-01T00:00Z/{year}-{month:02d}-28T23:30Z"
        r = requests.get(url)
        if r.status_code == 200:
            data = r.json().get("data", [])
            if data:
                frames.append(pd.DataFrame(data))
        else:
            print(f"Failed: {year}-{month:02d} (status {r.status_code})")

carbon_df = pd.concat(frames, ignore_index=True)
print(f"Carbon intensity: {carbon_df.shape}")
carbon_df.head()

Carbon intensity: (48237, 3)


,from,to,intensity
0,2020-12-31T23:30Z,2021-01-01T00:00Z,"{'forecast': 190, 'actual': 184, 'index': 'mod..."
1,2021-01-01T00:00Z,2021-01-01T00:30Z,"{'forecast': 181, 'actual': 187, 'index': 'mod..."
2,2021-01-01T00:30Z,2021-01-01T01:00Z,"{'forecast': 178, 'actual': 182, 'index': 'mod..."
3,2021-01-01T01:00Z,2021-01-01T01:30Z,"{'forecast': 175, 'actual': 178, 'index': 'mod..."
4,2021-01-01T01:30Z,2021-01-01T02:00Z,"{'forecast': 174, 'actual': 171, 'index': 'mod..."


In [12]:
# Skip this cell until ENTSO_API_KEY is set in .env
ENTSO_KEY = os.getenv("ENTSO_API_KEY")

if ENTSO_KEY:
    from entsoe import EntsoePandasClient

    client = EntsoePandasClient(api_key=ENTSO_KEY)
    start = pd.Timestamp("2021-01-01", tz="Europe/London")
    end = pd.Timestamp("2024-01-01", tz="Europe/London")

    gen = client.query_generation("GB", start=start, end=end, psr_type=None)
    gen.to_parquet(DATA_RAW / "generation_GB_2021_2024.parquet")
    print(f"Generation: {gen.shape}")

    load = client.query_load("GB", start=start, end=end)
    load.to_parquet(DATA_RAW / "load_GB_2021_2024.parquet")
    print(f"Load: {load.shape}")
else:
    print("ENTSO_API_KEY not found in .env — register at transparency.entsoe.eu and add the key, then rerun this cell.")

Generation: (7890, 11)
Load: (7886, 1)


In [13]:
carbon_df = pd.concat(
    [carbon_df.drop(columns=["intensity"]), carbon_df["intensity"].apply(pd.Series)],
    axis=1
)
carbon_df["from"] = pd.to_datetime(carbon_df["from"])
carbon_df["to"] = pd.to_datetime(carbon_df["to"])

carbon_df.to_parquet(DATA_RAW / "carbon_intensity_GB_2021_2024.parquet")
print("Saved:", DATA_RAW / "carbon_intensity_GB_2021_2024.parquet")
carbon_df.head()

Saved: ../data/raw/carbon_intensity_GB_2021_2024.parquet


,from,to,forecast,actual,index
0,2020-12-31 23:30:00+00:00,2021-01-01 00:00:00+00:00,190,184.0,moderate
1,2021-01-01 00:00:00+00:00,2021-01-01 00:30:00+00:00,181,187.0,moderate
2,2021-01-01 00:30:00+00:00,2021-01-01 01:00:00+00:00,178,182.0,moderate
3,2021-01-01 01:00:00+00:00,2021-01-01 01:30:00+00:00,175,178.0,moderate
4,2021-01-01 01:30:00+00:00,2021-01-01 02:00:00+00:00,174,171.0,moderate


In [6]:
import cdsapi
from pathlib import Path

c = cdsapi.Client()
DATA_RAW = Path("../data/raw")

for year in ['2021', '2022', '2023']:
    print(f"Requesting ERA5 for {year}...")
    c.retrieve(
        'reanalysis-era5-single-levels',
        {
            'product_type': 'reanalysis',
            'variable': [
                '10m_u_component_of_wind',
                '2m_temperature',
                'surface_solar_radiation_downwards',
            ],
            'year': [year],
            'month': [f'{m:02d}' for m in range(1, 13)],
            'day': [f'{d:02d}' for d in range(1, 32)],
            'time': ['00:00', '12:00'],
            'area': [61, -8, 49, 2],
            'format': 'netcdf',
        },
        str(DATA_RAW / f'era5_weather_GB_{year}.nc')
    )
    print(f"ERA5 {year} saved")

print("All ERA5 done")

Requesting ERA5 for 2021...


2026-07-09 20:21:22,590 INFO Request ID is 9148233a-2605-464b-914c-dcc38be2da18
2026-07-09 20:21:22,687 INFO status has been updated to accepted
2026-07-09 20:21:55,887 INFO status has been updated to running
2026-07-09 20:25:43,558 INFO status has been updated to successful


507f5e90e0d0fea5cb81a00959b51425.zip:   0%|          | 0.00/6.82M [00:00<?, ?B/s]

ERA5 2021 saved
Requesting ERA5 for 2022...


2026-07-09 20:25:49,015 INFO Request ID is 5b49731f-2053-48b3-84e9-878ef603b428
2026-07-09 20:25:50,078 INFO status has been updated to accepted
2026-07-09 20:26:11,488 INFO status has been updated to running
2026-07-09 20:28:54,602 INFO status has been updated to successful


133a086cdc2c9d5d8b638e11d0c2a062.zip:   0%|          | 0.00/6.81M [00:00<?, ?B/s]

ERA5 2022 saved
Requesting ERA5 for 2023...


2026-07-09 20:29:00,202 INFO Request ID is 2606ca85-152f-4ea0-9d19-02ba7436d730
2026-07-09 20:29:00,305 INFO status has been updated to accepted
2026-07-09 20:29:22,100 INFO status has been updated to running
2026-07-09 20:33:22,790 INFO status has been updated to successful


db6ff62ee44a7fcc9f6329974d92e5d2.zip:   0%|          | 0.00/6.80M [00:00<?, ?B/s]

ERA5 2023 saved
All ERA5 done


In [15]:
import requests
import pandas as pd
from pathlib import Path
import time
from datetime import date, timedelta

DATA_RAW = Path("../data/raw")

# Check if partial data exists already
existing_path = DATA_RAW / "elexon_fuelinst_2021_2023.parquet"
if existing_path.exists():
    print("Elexon file already exists - no need to rerun")
    elexon_df = pd.read_parquet(existing_path)
    print(f"Existing data shape: {elexon_df.shape}")
else:
    print("File not found - need to rerun")

Elexon file already exists - no need to rerun
Existing data shape: (5414080, 7)


In [16]:
import requests
import pandas as pd
from pathlib import Path
import time
from datetime import date, timedelta

DATA_RAW = Path("../data/raw")
frames = []
current = date(2021, 1, 1)
end_date = date(2023, 12, 31)

while current <= end_date:
    url = f"https://data.elexon.co.uk/bmrs/api/v1/datasets/FUELINST?settlementDate={current}&format=json"
    r = requests.get(url)
    if r.status_code == 200:
        data = r.json().get("data", [])
        if data:
            frames.append(pd.DataFrame(data))
    else:
        print(f"FAIL {current} status {r.status_code}")
    if current.day == 1:
        print(f"Progress: {current}")
    current += timedelta(days=1)
    time.sleep(0.3)

elexon_df = pd.concat(frames, ignore_index=True)
print(f"Elexon total: {elexon_df.shape}")
elexon_df.to_parquet(DATA_RAW / "elexon_fuelinst_2021_2023.parquet")
print("Saved elexon_fuelinst_2021_2023.parquet")

Progress: 2021-01-01
Progress: 2021-02-01
Progress: 2021-03-01
Progress: 2021-04-01
Progress: 2021-05-01
Progress: 2021-06-01
Progress: 2021-07-01
Progress: 2021-08-01
Progress: 2021-09-01
Progress: 2021-10-01
Progress: 2021-11-01
Progress: 2021-12-01
Progress: 2022-01-01


ReadTimeout: HTTPSConnectionPool(host='data.elexon.co.uk', port=443): Read timed out. (read timeout=None)